# FarmTech VisionGuard - Ir Além

## Opção 2: Transfer Learning + Fine Tuning + segmentação

Este notebook é **opcional** e não interfere na correção das Entregas 1 e 2.

Hipóteses:
1. Uma MobileNetV2 pré-treinada em ImageNet supera a CNN treinada do zero?
2. Remover o background por segmentação antes da classificação melhora o desempenho?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt, time
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

ROOT = Path('/content/drive/MyDrive/FarmTech_VisionGuard')
DATASET = ROOT/'dataset'

## Transfer Learning com MobileNetV2

In [ ]:
IMG=(224,224)
def paths_labels(split):
    paths=[]; labels=[]
    for p in sorted((DATASET/'images'/split).iterdir()):
        if p.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
        if p.stem.startswith('person_'): y=0
        elif p.stem.startswith('car_'): y=1
        else: continue
        paths.append(str(p)); labels.append(y)
    return paths, np.array(labels, dtype=np.float32)

def ds(paths, labels, training=False):
    x=tf.data.Dataset.from_tensor_slices((paths, labels))
    def load(p,y):
        im=tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        im=tf.image.resize(im, IMG)
        im=tf.keras.applications.mobilenet_v2.preprocess_input(im)
        return im,y
    if training: x=x.shuffle(len(paths), seed=42)
    return x.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(8).prefetch(tf.data.AUTOTUNE)

trp,try_=paths_labels('train'); vap,vay=paths_labels('val'); tep,tey=paths_labels('test')
train=ds(trp,try_,True); val=ds(vap,vay); test=ds(tep,tey)

base=tf.keras.applications.MobileNetV2(input_shape=(*IMG,3), include_top=False, weights='imagenet')
base.trainable=False
inp=tf.keras.Input(shape=(*IMG,3))
x=base(inp, training=False)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
x=tf.keras.layers.Dropout(.3)(x)
out=tf.keras.layers.Dense(1, activation='sigmoid')(x)
model=tf.keras.Model(inp,out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
history=model.fit(train, validation_data=val, epochs=12)

# Fine tuning das últimas 20 camadas
base.trainable=True
for layer in base.layers[:-20]:
    layer.trainable=False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
history_ft=model.fit(train, validation_data=val, epochs=10)

## Segmentação automática antes da classificação

A implementação abaixo usa DeepLabV3 pré-treinada para criar uma máscara semântica.
São mantidos pixels classificados como pessoa ou carro e o fundo é zerado.

> A disponibilidade/índices das classes dependem dos pesos do torchvision. A célula
> inspeciona os metadados dos pesos em vez de assumir índices fixos.

In [ ]:
import torch, torchvision
from PIL import Image
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

weights = DeepLabV3_ResNet50_Weights.DEFAULT
seg = deeplabv3_resnet50(weights=weights).eval().cuda() if torch.cuda.is_available() else deeplabv3_resnet50(weights=weights).eval()
preprocess = weights.transforms()
categories = list(weights.meta['categories'])
print(categories)

target_indices = {i for i,n in enumerate(categories) if n.lower() in {'person','car'}}

def mask_image(path):
    image=Image.open(path).convert('RGB')
    x=preprocess(image).unsqueeze(0)
    if torch.cuda.is_available(): x=x.cuda()
    with torch.no_grad():
        pred=seg(x)['out'][0].argmax(0).cpu().numpy()
    mask=np.isin(pred, list(target_indices)).astype(np.uint8)
    arr=np.asarray(image.resize((mask.shape[1],mask.shape[0])))
    masked=arr * mask[...,None]
    return image, mask, Image.fromarray(masked.astype(np.uint8))

orig, mask, masked = mask_image(tep[0])
fig, axes = plt.subplots(1,3,figsize=(14,5))
axes[0].imshow(orig); axes[0].set_title('Original')
axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Máscara')
axes[2].imshow(masked); axes[2].set_title('Background removido')
for ax in axes: ax.axis('off')
plt.show()

## Avaliação crítica esperada

Após executar, compare:
- CNN do zero;
- MobileNetV2 congelada;
- MobileNetV2 com Fine Tuning;
- classificação usando imagens mascaradas.

Discuta acurácia, tempo de treinamento, tempo de inferência e custo de implementação.
Não conclua que segmentação "melhora" antes de medir.

## Avaliação quantitativa do Transfer Learning

A avaliação abaixo mede o modelo após o Fine Tuning usando exatamente as 8 imagens de teste.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from time import perf_counter

start = perf_counter()
probs_original = model.predict(test, verbose=0).ravel()
infer_original_s = perf_counter() - start
pred_original = (probs_original >= 0.5).astype(int)

acc_o = accuracy_score(tey, pred_original)
p_o, r_o, f1_o, _ = precision_recall_fscore_support(
    tey, pred_original, average='macro', zero_division=0
)
original_metrics = {
    'accuracy': float(acc_o),
    'macro_precision': float(p_o),
    'macro_recall': float(r_o),
    'macro_f1': float(f1_o),
    'mean_inference_ms': float(infer_original_s / len(tey) * 1000),
}
original_metrics

## Criar dataset segmentado automaticamente

A rede DeepLabV3 cria uma máscara para `person` e `car`. O fundo é removido e as
imagens resultantes são usadas em um novo treinamento com a **mesma arquitetura
MobileNetV2**, permitindo comparar imagem original versus imagem pré-segmentada.

In [ ]:
from pathlib import Path
import shutil

MASKED = Path('/content/farmtech_masked')
if MASKED.exists():
    shutil.rmtree(MASKED)

def create_masked_dataset():
    for split in ('train', 'val', 'test'):
        src_dir = DATASET/'images'/split
        dst_dir = MASKED/'images'/split
        dst_dir.mkdir(parents=True, exist_ok=True)
        for path in sorted(src_dir.iterdir()):
            if path.suffix.lower() not in {'.jpg','.jpeg','.png'}:
                continue
            _, _, masked_image = mask_image(path)
            masked_image.save(dst_dir/path.name)

create_masked_dataset()
print('Dataset segmentado:', MASKED)

In [ ]:
def paths_labels_root(root, split):
    paths=[]; labels=[]
    for p in sorted((Path(root)/'images'/split).iterdir()):
        if p.suffix.lower() not in {'.jpg','.jpeg','.png'}:
            continue
        if p.stem.startswith('person_'): y=0
        elif p.stem.startswith('car_'): y=1
        else: continue
        paths.append(str(p)); labels.append(y)
    return paths, np.array(labels, dtype=np.float32)

mtrp,mtry_=paths_labels_root(MASKED,'train')
mvap,mvay=paths_labels_root(MASKED,'val')
mtep,mtey=paths_labels_root(MASKED,'test')
mtrain=ds(mtrp,mtry_,True); mval=ds(mvap,mvay); mtest=ds(mtep,mtey)

base_m=tf.keras.applications.MobileNetV2(
    input_shape=(*IMG,3), include_top=False, weights='imagenet'
)
base_m.trainable=False
inp_m=tf.keras.Input(shape=(*IMG,3))
x_m=base_m(inp_m, training=False)
x_m=tf.keras.layers.GlobalAveragePooling2D()(x_m)
x_m=tf.keras.layers.Dropout(.3)(x_m)
out_m=tf.keras.layers.Dense(1, activation='sigmoid')(x_m)
masked_model=tf.keras.Model(inp_m,out_m)
masked_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

start=time.perf_counter()
masked_model.fit(mtrain, validation_data=mval, epochs=12, verbose=1)
base_m.trainable=True
for layer in base_m.layers[:-20]:
    layer.trainable=False
masked_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
masked_model.fit(mtrain, validation_data=mval, epochs=10, verbose=1)
masked_train_seconds=time.perf_counter()-start

In [ ]:
import pandas as pd

start=perf_counter()
probs_masked=masked_model.predict(mtest, verbose=0).ravel()
infer_masked_s=perf_counter()-start
pred_masked=(probs_masked>=0.5).astype(int)

acc_m=accuracy_score(mtey,pred_masked)
p_m,r_m,f1_m,_=precision_recall_fscore_support(
    mtey,pred_masked,average='macro',zero_division=0
)

masked_metrics={
    'accuracy':float(acc_m),
    'macro_precision':float(p_m),
    'macro_recall':float(r_m),
    'macro_f1':float(f1_m),
    'mean_inference_ms':float(infer_masked_s/len(mtey)*1000),
    'training_seconds':float(masked_train_seconds),
}

comparison_bonus=pd.DataFrame([
    {'abordagem':'MobileNetV2 + Fine Tuning - original', **original_metrics},
    {'abordagem':'MobileNetV2 + Fine Tuning - segmentada', **masked_metrics},
])
comparison_bonus

## Conclusão do "Ir Além"

Depois da execução, interprete a tabela sem assumir previamente que a segmentação ajuda.
Se o modelo segmentado superar o original, discuta a redução de ruído de background.
Se piorar, discuta perda de contexto, erros da máscara e domínio diferente da rede de
segmentação. Compare ainda o Transfer Learning com a CNN do zero do notebook obrigatório.